[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/jeffery/jeffery.ipynb)

# Jeffery Orbits

This example computes Jeffery orbits, the torque-free tumbling of rigid ellipsoids carried by a slow, viscous shear flow. Grains of rice stirred into a jar of honey align nearly parallel to the shear, linger there, and then flip end over end on a repeating schedule. Because a rigid particle cannot stretch with the surrounding fluid, viscous stresses on its surface select the unique angular rate that leaves zero net torque on the body: a sphere turns steadily with the fluid, a prolate rod returns along a closed orbit on the sphere of directions, and a triaxial ellipsoid precesses without closing.

The ambient velocity gradient is modelled as a linear map on offsets, the body's shape as its dual quadric `stretch(Plane.dual()(stretch.adjugate()))` sending planes to normal offsets, and the body's turning rate as a bivector whose rigid velocity map is `Point.commutator(rate)`. Torque balance reduces to a single map identity, `Point.commutator(rate)(shape) + shape(Plane.commutator(rate)) == gradient(shape) - shape(gradient.adjugate())`, equating the dual quadric's rigid rate of change to the skew action of the flow gradient against its adjugate. Because both maps are antisymmetric, contracting them with the regressive join `(Point & map(Plane)).trace(1, 2)` extracts all three independent components without a metric, yielding the turning rate at any orientation in one linear solve.

In Gibbs dyadic suspension mechanics, the unit director of an axisymmetric spheroid obeys the equation derived by [Jeffery in 1922](https://doi.org/10.1098/rspa.1922.0078), $\dot{\mathbf{p}} = \boldsymbol{\Omega}\times\mathbf{p} + \lambda(\mathbf{E}\cdot\mathbf{p} - (\mathbf{p}\cdot\mathbf{E}\cdot\mathbf{p})\mathbf{p})$ with Bretherton shape factor $\lambda = (r^2 - 1)/(r^2 + 1)$, subtracting the radial projection $(\mathbf{p}\cdot\mathbf{E}\cdot\mathbf{p})\mathbf{p}$ to preserve $|\mathbf{p}| = 1$ and tracking orbit phase through the Bretherton constant $C = p_y^{-1}\sqrt{p_x^2 + r^2 p_y^2}$ (or $p_z$ in the shear-vorticity convention), whereas a general triaxial ellipsoid requires transforming $\mathbf{E}$ and $\boldsymbol{\Omega}$ into the co-rotating principal frame to evaluate $\omega_{ij} - \Omega_{ij} = \frac{a_i^2 - a_j^2}{a_i^2 + a_j^2}\,E_{ij}$.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.util.animation import save_figure, show_states
from examples.fluids.jeffery import render

ga = VGA3D
mv = NumpyContext(ga).multivector
# Offsets and velocities are antivectors with intrinsic orientation.
Point = ga.gatype.antivector()                             # Antivector-
Rate = ga.gatype.bivector()                                # Bivector+
Rotor = ga.gatype.rotor()                                  # Rotor+
# Planes that read offsets are vectors with extrinsic orientation.
Plane = ga.gatype.vector()                                 # Vector+
# Linear stretch carrying the unit sphere onto the body.
Stretch = ga.gatype((Point, Point))                        # Point- <- Point-
# Dual quadric of the body.
Shape = ga.gatype((Point, Plane))                          # Point- <- Plane+
# Velocity gradient of the ambient flow.
Gradient = ga.gatype((Point, Point))                       # Point- <- Point-

axes = mv.basis()                                          # [3] Plane+
x, y, z = axes                                             # [] Plane+, Plane+, Plane+

## Ambient shear flow and ellipsoid dual quadrics

In simple shear between parallel plates, velocity along the flow direction grows linearly with height `y & Point` across the gradient direction. Half the vorticity bivector of the velocity gradient gives the ambient fluid's own turning rate. Each rigid particle stretches the unit sphere along its three principal axes, carrying the sphere's duality map `Plane.dual()` into the dual quadric `stretch(Plane.dual()(stretch.adjugate()))` whose surface offsets satisfy `shape.solve(offset) & offset == 1`.

In [ ]:
# Principal axis ratios of a sphere, a prolate rod, and a triaxial ellipsoid of equal volume.
axis_ratios = np.array([[1, 1, 1], [4, 1, 1], [4, 2, 1]], dtype=float)
equal_volume_radius = 1.3
semi_axes = equal_volume_radius * axis_ratios / np.prod(axis_ratios, axis=-1, keepdims=True) ** (1 / 3)
rod_aspect = axis_ratios[1, 0] / axis_ratios[1, 1]
shear_rate = 1.0

In [ ]:
def flow_curl(gradient: Gradient) -> Rate:
    """Vorticity bivector whose commutator action equals the velocity gradient minus its metric adjoint."""
    # The reciprocal of an orthonormal antivector basis is its reverse; the commutator spans the turning plane.
    return Point.reverse().commutator(gradient(Point)).contract(1, 2)          # [...] Rate+


flow_gradient = shear_rate * x.dual() * (y & Point)                            # [] Point- <- Point-
fluid_spin = flow_curl(flow_gradient) / 2                                      # [] Rate+
stretches = (axes.dual() * semi_axes * (axes & Point)).sum(axis=-1)            # [3] Point- <- Point-


def body_shapes(stretches: Stretch, rotors: Rotor) -> Shape:
    """Dual quadrics of ellipsoids with principal stretches turned by their orientation rotors."""
    turned_stretch = rotors >> stretches(rotors << Point)                      # [...] Point- <- Point-
    # Metric roundness enters once through the unit sphere's duality map from planes to normal offsets.
    unit_sphere = Plane.dual()                                                 # [] Point- <- Plane+
    return turned_stretch(unit_sphere(turned_stretch.adjugate()))              # [...] Point- <- Plane+

## Torque-free turning rate

Under a turning rate bivector, every offset of the rigid body moves at `offset.commutator(rate)` while every plane reading it turns at `plane.commutator(rate)`. Setting the resulting rate of change of the dual quadric equal to the skew flow action `gradient(shape) - shape(gradient.adjugate())` yields an antisymmetric map equation on planes; pairing its output with a free offset by `&` and taking the trace isolates all three components for a direct linear solve. For a sphere, the dual quadric is isotropic and the particle turns at the ambient fluid's own spin.

In [ ]:
def jeffery_spin(shape: Shape, gradient: Gradient) -> Rate:
    """Torque-free turning rate of a rigid ellipsoid of the given dual quadric in a linear velocity field."""
    # Rate of change of the dual quadric under rigid turning of both its output offsets and input planes.
    turning_rate = Point.commutator(Rate)(shape) + shape(Plane.commutator(Rate))   # [...] Point- <- (Plane+, Rate+)
    # Skew action of the ambient velocity gradient on offsets against its adjugate on planes.
    flow_rate = gradient(shape) - shape(gradient.adjugate())                       # [...] Point- <- Plane+
    # Contracting each skew map with the regressive join isolates its three independent bivector-like components.
    return (Point & turning_rate(Plane)).trace(1, 2).solve((Point & flow_rate(Plane)).trace(1, 2))   # [...] Rate+


sphere_spin = jeffery_spin(body_shapes(stretches[0], mv.rotor()), flow_gradient)   # [] Rate+

## Tumbling in shear flow and direction-sphere orbits

Stepping all three bodies across five initial tilts from the vortex normal advances the full batch together by midpoint rotor integration, while streaming a single initial tilt directly to the ray-cast shear scene. This animation cell takes about five seconds to trace the octant-shaded ellipsoids and render the frames.

In [ ]:
initial_tilts = np.array([0.15, 0.4, 0.7, 1.0, 1.3])
shown_start = 2
n_periods = 2
n_steps_per_period = 120
n_steps = n_periods * n_steps_per_period
n_steps_per_frame = 2
n_pixels = 240
heights = y.dual() * np.array([-3, -2, -1, 1, 2, 3], dtype=float)              # [6] Point-
frame_duration_ms = 40

In [ ]:
def midpoint_step(rotors: Rotor, dt: float) -> Rotor:
    """Orientation rotors advanced across a timestep `dt` by the midpoint Jeffery spin."""
    halfway_rotors = (jeffery_spin(body_shapes(stretches[:, None], rotors), flow_gradient) * (-dt / 4)).exp() * rotors   # [3, 5] Rotor+
    return (jeffery_spin(body_shapes(stretches[:, None], halfway_rotors), flow_gradient) * (-dt / 2)).exp() * rotors     # [3, 5] Rotor+


def tumbling_stream(rotors: Rotor, dt: float, n_steps: int):
    """Yield orientation rotors from the initial state across `n_steps` midpoint steps."""
    yield rotors
    for _ in range(n_steps):
        rotors = midpoint_step(rotors, dt)
        yield rotors


def animate_shown_start(stretches: Stretch, heights: Point, velocities: Point, n_pixels: int, step_stream):
    """Render strided single-start orientations and long-axis tips from the full batch rotor stream."""
    shown_states = (
        (rotors[:, shown_start], rotors[:, shown_start] >> (x.dual() * semi_axes[:, 0]))
        for step_index, rotors in enumerate(step_stream)
        if step_index % n_steps_per_frame == 0
    )
    return render.animate_tumbling(stretches, heights, velocities, n_pixels, shown_states)


jeffery_period = 2 * np.pi * (rod_aspect + 1 / rod_aspect) / shear_rate
start_rotors = ((x ^ z) * (-(np.pi / 2 - initial_tilts) / 2)).exp()            # [5] Rotor+
initial_rotors = start_rotors[None].broadcast_to((len(semi_axes), len(initial_tilts)))   # [3, 5] Rotor+
rotor_steps = show_states(
    tumbling_stream(initial_rotors, jeffery_period / n_steps_per_period, n_steps),
    animate_shown_start,
    stretches,
    heights,
    flow_gradient(heights),
    n_pixels,
    name="jeffery_tumbling",
    duration_ms=frame_duration_ms,
);

Behind the three octant-painted bodies, the grey arrows show the ambient shear velocity profile and the orange curves trace the paths swept by both ends of each body's long axis.

Projecting both antipodal ends of the long axis across all five initial tilts onto the unit sphere of directions separates the three regimes cleanly.

In [ ]:
rotors = stack(rotor_steps)                                                    # [n_steps + 1, 3, 5] Rotor+
trails = rotors >> x.dual()                                                    # [n_steps + 1, 3, 5] Point-
save_figure(render.draw_orbits(trails), "jeffery_orbits");

From left to right, the panels show the sphere, the prolate rod, and the triaxial ellipsoid, with grey arrows marking the flow direction `x`, velocity-gradient direction `y`, and vortex normal `z`, and viridis curves colouring the five initial tilts. On the sphere, every axis turns at a uniform rate on a latitude circle; on the prolate rod, every orbit closes after one Jeffery period while lingering near the flow plane; on the triaxial ellipsoid, coupling among three unequal axes breaks orbit closure.

In [ ]:
# checks
assert ((rotors.scalar_norm_squared() - 1).abs() < 1e-11).all()
assert ((sphere_spin - fluid_spin).norm() < 1e-12).all()
rod_trails = trails[:, 1]                                                      # [n_steps + 1, 5] Point-
assert ((rod_trails[n_steps_per_period] - rod_trails[0]).norm() < 2e-2).all()
flow_coord, gradient_coord, vortex_coord = x & rod_trails, y & rod_trails, z & rod_trails   # [n_steps + 1, 5] Scalar+, Scalar+, Scalar+
orbit_constant = (flow_coord.squared() + rod_aspect**2 * gradient_coord.squared()) / (rod_aspect**2 * vortex_coord.squared())
assert ((orbit_constant / orbit_constant[0] - 1).abs() < 2e-2).all()